# causomic User Manual

This notebook reviews the main functions and workflow in causomic.

causomic is designed to perform causal inference on Mass Spectrometry 
(MS)-based proteomics experiments. The goal is to predict the effect of 
interventions (e.g., inhibition) on the biological system.

## General imports

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
%matplotlib inline

## Input data

`causomic` takes as input the output of the `MSstats` `dataProcess` function.
The `dataProcess` function outputs a list of mutitple `data.frame`. 
Specifically, causomic takes as input the `ProteinLevelData` `data.frame` 
output of `dataProcess`. The `ProteinLevelData` is the summarized protein-level 
data, which includes one value per protein per MS run. This is contrast to 
the actual output of MS-based experiments (generally either the precursor or 
fragment, depending on the acquisition method / MS1 vs MS2 quantification).

`MSstats` is an R package which can be used to process identified and 
quantified MS-based proteomics data. Eventually the goal would be to call this 
function directly from `causomic`. For now, you can find R code to run this 
function in the notes of the `causomic.data_analysis.dataProcess` function.

In this example we will use the small molecule inhibition dataset from Talus 
Bio.

In [ ]:
input_data = pd.read_csv("../data/Talus/processed_data/ProteinLevelData.csv")
input_data.head()

## Data pre-processing

`MSstats` will already take care of most data processing for us, however the 
output of the function is the log normalized intensities. The scale of these 
could be an issue. To alleviate this we can just zero center the values using 
the `normalize` function.

In [ ]:
import causomic.data_analysis.normalization as norm

normalized_input_data = norm.normalize(input_data)
normalized_input_data.head()

The last step of data processing is to convert the data into wide format. This 
is the expected input for the causal inference model.

In [ ]:
import causomic.data_analysis.gene_set as gs # TODO: This function should be moved to the data_analysis module

input_data_wide = gs.prep_msstats_data(normalized_input_data, 
                                  gene_map=None, 
                                  parse_gene=True)

# TODO: This should be fixed in the prep_msstats_data function
input_data_wide = input_data_wide.reset_index(drop=True)
input_data_wide.columns.name = None

input_data_wide.head()

## Gene set correlation analysis

A user may be interested in performing a correlation analysis as part of the 
analysis pipeline. This is particularly helpful when trying to determine 
what gene sets might be of interest to build a network around. `causomic` 
includes a function to determine all pairwise correlations between proteins 
in the dataset.

In [ ]:
import causomic.data_analysis.gene_set as gs # TODO: This function should be moved to the data_analysis module

corr_data = gs.gen_correlation_matrix(input_data_wide, methods=["pearson"], abs_corr=True)

In [ ]:
corr_data["pearson"].head()

## Gene set analysis

`causomic` includes functionality for gene set analysis, in case a user 
wants to analyze specific pathways of interest.

This function takes as input a list of gene sets from MSigDB. It looks at how 
many of the proteins in each gene set are highly correlated.

Additionally, the function can see how many differentially abundant proteins are
in the gene set, if differential analysis results are provided (optional).

In [ ]:
import causomic.data_analysis.gene_set as gs

regulatory_paths = gs.test_gene_sets(corr_data, # Correlation data
                                     input_data.columns.values, # All protein names
                                     "../data/gene_sets/test_pathways.json", # MSigDB gene sets
                                     threshold=0.33)# Threshold for correlation

In [ ]:
regulatory_paths.head()

When we find a gene set of interest, we can extract the genes in the pathway 
using the `extract_genes_in_path` function.

In [ ]:
example_pathway = gs.extract_genes_in_path(
    input_data_wide.columns, # All protein names
    "ARHGAP35_TARGET_GENES", # Pathway name of interest
    "../data/gene_sets/test_pathways.json", # MSigDB gene sets
    return_all=False) # Return all genes in the pathway or only measured proteins

example_pathway

Alternatively, we could look for gene sets which include specific genes of 
interest. This is done with the `find_sets_with_gene` function.

In [ ]:
reg_sets = gs.find_sets_with_gene(["BRD2"], 
                                  "../data/gene_sets/test_pathways.json")
reg_sets

## Pull network from INDRA

In this section we will pull a network from the biological knowledge database 
INDRA. In order for this to work you must have a `.env` file located in the root
of the `causomic` with the URL and login information stored. Additionally, 
you must be on Northeastern University's IP address. In the future this needs to
be changed for external users.

The first step is to build a the Neo4jClient using the information in the `.env`
file.

In [ ]:
from indra_cogex.client import Neo4jClient # TODO: Move this to the analysis_uniprot function

client = Neo4jClient(url=os.getenv("API_URL"), 
                        auth=(os.getenv("USER"), 
                            os.getenv("PASSWORD"))
                    )

Now we can pull a network using the `analysis_uniprot` function. 

**Note**: Right now this function is a bit of a mess and relies on an edited 
version of `indra_cogex`. I am working to move the edited versions into 
`causomic`. 

In [ ]:
from causomic.graph_construction.indra_networks import analysis_uniprot

set_name = "ARHGAP35_TARGET_GENES"
indra_network = analysis_uniprot(
    ids=example_pathway,
    analysis_id=set_name,
    client=client,
    minimum_evidence_count=1,
    id_type="gene"
)

In [ ]:
indra_network.head()

## Construct latent variable network

The graph extracted from INDRA will inherently be messy and unusable for classic
causal inference. In this section we use the `GraphBuilder` object to convert 
the INDRA statements into a LVM which can be used for causal inference.

In [ ]:
from causomic.graph_construction.graph import GraphBuilder

example_graph = GraphBuilder(indra_network, 
                                normalized_input_data, 
                                True) # MSstats format or not

example_graph.build_full_graph(data_type="LF", # LF (label free) or TMT
                        protein_format="Gene_Name_Organism", # How the proteins are named in the data
                        source_name="source_hgnc_symbol",
                        target_name="target_hgnc_symbol")
example_graph.build_dag() # TODO: Right now removing cycles can take forever on long graphs. Need a better optimized algo for this.
example_graph.create_latent_graph()
example_graph.plot_latent_graph(figure_size=(12, 8))

## Fit LVM

Now we have a graph and data. We are ready to do some causal inference.

In [ ]:
from causomic.causal_model.LVM import LVM
import pyro

pyro.clear_param_store() # In case you want to run multiple models
lvm = LVM(example_graph.experimental_data.reset_index(drop=True), # Data is already in graph object
          example_graph.causal_graph)
lvm.prepare_graph()
lvm.prepare_data()

lvm.fit_model(num_steps=5000) # Num steps doesn't matter, will be stopped by convergence

With the trained model we can now perform an intervention.

In [ ]:
lvm.intervention({"KEAP1": -5}, "CUL3")
int1 = lvm.intervention_samples
int2 = lvm.posterior_samples

In [ ]:
fig, ax = plt.subplots()

ax.hist(int1, alpha=.5, color="blue", bins=20, label="intervention")
ax.hist(int2, alpha=.5, color="orange", bins=20, label="ground truth")
plt.legend()